# Lab 0-03 Assignment: Revise a Prompt for More Consistent Results

Use this notebook after you finish `02_model_comparison.ipynb`.

What stays the same from `02_model_comparison.ipynb`:
- the same three class models
- the same synthetic case note
- the same baseline prompt structure
- the same basic workflow of running the models and reviewing raw outputs

What is new in this notebook:
- you revise the prompt yourself
- you rerun the same three models with the revised prompt
- you compare the before/after results with a simple summary

In this assignment, you will:
- use the same three class models from `02_model_comparison.ipynb`
- run the original baseline prompt on those three models
- identify one inconsistency in the results
- revise the prompt to make the outputs more consistent
- rerun the same three models
- compare the before/after results with a simple summary

## Follow the Story

Three AI assistants read the same synthetic forensic case note. They use `qwen3.5:0.8b`, `qwen3.5:27b`, and `gemma4:e4b`. First, they answer with the original prompt. Then you improve the prompt and ask the same three assistants again.

### Goal

Learn how a clearer prompt can make AI answers more consistent. You will compare the before and after answers, especially how each assistant writes device IDs. The notebook does not choose the best prompt for you; it gives you the evidence to explain what changed and why.

## Step 0: Check Your Setup

Run the next code block first. It confirms that the notebook is open from the correct lab folder, reads the `.env` settings file, and prepares the connection to Ollama. The first real contact with Ollama happens in Step 1.

In [1]:
import json
from html import escape
from pathlib import Path
from time import perf_counter

import requests
from dotenv import dotenv_values
from IPython.display import HTML, display
from openai import OpenAI

# WHAT THIS BLOCK DOES: Prepare the notebook for the comparison.
# It checks the lab folder, reads the settings file, and creates the object
# used to talk to the local Ollama service.
LAB_NAME = 'lab0_03_model_warmup'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

config = dotenv_values(env_path)
# This test model confirms that the .env settings were read correctly.
# The three fixed class models below are used for the actual comparison.
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print("Test model from this lab's .env:", default_model)
print('Ollama address:', ollama_base_url)


Repo root: /home/frank/projects/agentic-AI4-forensics
Lab folder: /home/frank/projects/agentic-AI4-forensics/lab0_03_model_warmup
Test model from this lab's .env: qwen3:8b
Ollama address: http://localhost:11434/v1


## Step 1: Discover Available Models

Run the next block to take a roll call of the models available from Ollama.

In [2]:
# WHAT THIS BLOCK DOES: Ask Ollama which models are ready to use.
# The list lets the next step confirm that the three class models are installed.
tags_url = ollama_base_url.rstrip('/').replace('/v1', '/api/tags')

response = requests.get(tags_url, timeout=10)
response.raise_for_status()
available_models = [item.get('name') for item in response.json().get('models', []) if item.get('name')]

print('Available models:')
for index, model_name in enumerate(available_models, start=1):
    print(f'{index}. {model_name}')

if len(available_models) < 3:
    raise ValueError('This assignment needs at least 3 available models.')


Available models:
1. qwen3-vl:latest
2. kimi-k2.5:cloud
3. kimi-k2.6:cloud
4. qwen3.6:latest
5. gemma4:31b
6. gemma4:e4b
7. huihui_ai/qwen3.5-abliterated:latest
8. qwen3.5:2b
9. qwen3.5:9b
10. qwen3.5:0.8b
11. qwen3.5:35b-a3b
12. qwen3:8b
13. qwen3.5:27b
14. llama3.3:70b


## Step 2: Confirm Three Comparison Models

Use the same three class models from `02_model_comparison.ipynb` so everyone in the class works with the same set:
- `qwen3.5:0.8b`
- `qwen3.5:27b`
- `gemma4:e4b`

The next block checks that all three are available from Ollama.

In [3]:
# WHAT THIS BLOCK DOES: Keep every student comparison on the same three models.
models_to_compare = ['qwen3.5:0.8b', 'qwen3.5:27b', 'gemma4:e4b']

missing_models = [name for name in models_to_compare if name not in available_models]

print('Models selected for comparison:')
for model_name in models_to_compare:
    print('-', model_name)

if missing_models:
    raise ValueError(f'These required models are missing from Ollama: {missing_models}')

if len(set(models_to_compare)) != 3:
    raise ValueError('Choose 3 different models before continuing.')


Models selected for comparison:
- qwen3.5:0.8b
- qwen3.5:27b
- gemma4:e4b


## Step 3: Synthetic Case Note

Use the same synthetic note for both the baseline prompt and your revised prompt.

In [4]:
case_note = """
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.
""".strip()

print(case_note)

Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.


## Step 4: Run the Baseline Comparison

### 4.1 Create the Baseline Prompt

Run the original prompt first so you have a baseline for comparison.

In [5]:
baseline_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

print(baseline_prompt)

Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.


### 4.2 Ask All Three Models

This cell sends the original prompt to each of your three models and prints their raw outputs.

In [6]:
# This helper removes code boxes or extra words around a model's JSON answer.
# That makes the before and after answers easier to compare.
def clean_json_text(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned

# This helper asks one model the prompt and keeps its time, full answer,
# and JSON-format result together.
def ask_model(model_name: str, prompt: str) -> dict:
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
    )
    elapsed = perf_counter() - start
    raw_text = response.choices[0].message.content
    cleaned_text = clean_json_text(raw_text)

    try:
        parsed = json.loads(cleaned_text)
        parse_error = None
    except Exception as exc:
        parsed = None
        parse_error = str(exc)

    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
        'parsed': parsed,
        'parse_error': parse_error,
    }

# This helper gives the same prompt to all three models.
# Using the same models in both runs makes the before and after comparison fair.
def run_prompt_for_models(prompt_name: str, prompt: str) -> list[dict]:
    results = []
    for model_name in models_to_compare:
        print(f'Running {prompt_name} prompt on {model_name}...')
        result = ask_model(model_name, prompt)
        result['prompt_name'] = prompt_name
        results.append(result)
    return results

baseline_results = run_prompt_for_models('baseline', baseline_prompt)

print('\nBaseline raw outputs:')
for result in baseline_results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()


Running baseline prompt on qwen3.5:0.8b...
Running baseline prompt on qwen3.5:27b...
Running baseline prompt on gemma4:e4b...

Baseline raw outputs:
Model: qwen3.5:0.8b
Time (seconds): 57.61
Parse error: None
--------------------------------------------------------------------------------
{
  "names": [
    "investigator maya chen",
    "jordan lee",
    "priya nair"
  ],
  "phone_numbers": [
    "415-555-0187",
    "202-555-0142"
  ],
  "email_addresses": [
    "alex.romero88@example.com",
    "priyanair@sample.org"
  ],
  "physical_addresses": [
    "2458 West Pine Street Springfield, IL 62704"
  ],
  "device_ids": [
    "356938035643809",
    "SN-A19XZ-4421"
  ]
}

Model: qwen3.5:27b
Time (seconds): 114.93
Parse error: None
--------------------------------------------------------------------------------
{
  "names": [
    "Maya Chen",
    "Jordan Lee",
    "Priya Nair"
  ],
  "phone_numbers": [
    "415-555-0187",
    "202-555-0142"
  ],
  "email_addresses": [
    "alex.romero88@exa

## Step 5: Revise the Prompt

### 5.1 Create Your Revised Prompt

Edit the prompt below before running the next cell.

Goal: make the results more consistent across models.

A good place to focus is `device_ids`, because the baseline prompt does not clearly say whether labels such as `IMEI` should be included.

In [ ]:
# WHAT THIS BLOCK DOES: Give you a copy of the baseline prompt to improve.
# Add at least one rule that makes device ID output more consistent.
# Do not include labels, such as "IMEI" before extracted device_ids.as "IMEI".

revised_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

if revised_prompt == baseline_prompt:
    print('Reminder: revise the prompt before running Step 6.')

print(revised_prompt)


Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.
- Do not include labels, such as "IMEI" before extracted device_ids.

Case note:
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.


## Step 6: Run the Revised Prompt and Compare the Results

### 6.1 Ask All Three Models Again

This code reruns the same three models with your revised prompt. It first prints the full revised answers so you can read them before looking at the short comparison table.

### 6.2 Compare Before and After

The table compares each model's baseline and revised response time, JSON format, and device IDs.

Use the full answers and table together when you decide whether your prompt revision helped.

In [8]:
revised_results = run_prompt_for_models('revised', revised_prompt)

print('\nRevised raw outputs:')
for result in revised_results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()

baseline_summary = []
for result in baseline_results:
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else None
    keys_found = sorted(parsed.keys()) if isinstance(parsed, dict) else []
    device_ids_found = parsed.get('device_ids', []) if isinstance(parsed, dict) else []
    baseline_summary.append({
        'model': result['model'],
        'seconds': result['seconds'],
        'valid_json': parsed is not None,
        'keys_found': keys_found,
        'device_ids_found': device_ids_found,
    })

revised_summary = []
for result in revised_results:
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else None
    keys_found = sorted(parsed.keys()) if isinstance(parsed, dict) else []
    device_ids_found = parsed.get('device_ids', []) if isinstance(parsed, dict) else []
    revised_summary.append({
        'model': result['model'],
        'seconds': result['seconds'],
        'valid_json': parsed is not None,
        'keys_found': keys_found,
        'device_ids_found': device_ids_found,
    })

comparison_by_model = []
for model_name in models_to_compare:
    baseline_item = next(item for item in baseline_summary if item['model'] == model_name)
    revised_item = next(item for item in revised_summary if item['model'] == model_name)
    comparison_by_model.append({
        'model': model_name,
        'baseline': baseline_item,
        'revised': revised_item,
    })

# Turn a list of device IDs into text that fits inside one table cell.
def table_items(items):
    return '<br>'.join(escape(str(item)) for item in items) if items else '—'

rows = []
for item in comparison_by_model:
    baseline, revised = item['baseline'], item['revised']
    rows.append(f"""
        <tr>
          <td>{escape(item['model'])}</td>
          <td>{baseline['seconds']}</td>
          <td>{'Yes' if baseline['valid_json'] else 'No'}</td>
          <td>{table_items(baseline['device_ids_found'])}</td>
          <td>{revised['seconds']}</td>
          <td>{'Yes' if revised['valid_json'] else 'No'}</td>
          <td>{table_items(revised['device_ids_found'])}</td>
        </tr>
    """)

table_html = """
<table border='1' cellpadding='6' style='border-collapse:collapse'>
  <tr>
    <th rowspan='2'>Model</th><th colspan='3'>Baseline prompt</th><th colspan='3'>Revised prompt</th>
  </tr>
  <tr>
    <th>Seconds</th><th>Valid JSON?</th><th>Device IDs</th>
    <th>Seconds</th><th>Valid JSON?</th><th>Device IDs</th>
  </tr>
  {rows}
</table>
""".format(rows=''.join(rows))

display(HTML(table_html))

Running revised prompt on qwen3.5:0.8b...
Running revised prompt on qwen3.5:27b...
Running revised prompt on gemma4:e4b...

Revised raw outputs:
Model: qwen3.5:0.8b
Time (seconds): 41.38
Parse error: None
--------------------------------------------------------------------------------
{
  "names": [
    "Jordan Lee"
  ],
  "phone_numbers": [
    "415-555-0187",
    "202-555-0142"
  ],
  "email_addresses": [
    "alex.romero88@example.com",
    "priyanair@sample.org"
  ],
  "physical_addresses": [
    "2458 West Pine Street, Springfield, IL 62704"
  ],
  "device_ids": [
    "356938035643809",
    "SN-A19XZ-4421"
  ]
}

Model: qwen3.5:27b
Time (seconds): 85.46
Parse error: None
--------------------------------------------------------------------------------
{
  "names": [
    "Maya Chen",
    "Jordan Lee",
    "Priya Nair"
  ],
  "phone_numbers": [
    "415-555-0187",
    "202-555-0142"
  ],
  "email_addresses": [
    "alex.romero88@example.com",
    "priyanair@sample.org"
  ],
  "physic

## Step 7: Reflection Questions

Replace this text with short answers to the questions below.

Use the before/after summary and the raw outputs to support your answers.

1. What inconsistency did you notice in the baseline outputs?
2. What specific rule did you add to the revised prompt?
3. Did the revised prompt make the `device_ids` format more consistent? Give one example.
4. Did JSON validity or returned keys change from baseline to revised?
5. Which model changed the most after your prompt revision, and what changed?

## Step 8: Submission

Save the notebook with:
- the three selected models
- the baseline prompt and baseline raw outputs
- your revised prompt
- the revised raw outputs
- the summary comparison
- your short reflection